<a href="https://colab.research.google.com/github/skandurigit/Langchain_Tutorial/blob/main/session_1_hands_on_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Fundamentals with LangChain — Hands-On
### Agentic AI Enterprise Mastery Bootcamp · Session 1 (Part 2)

Welcome to your **first hands-on**. Today you'll *watch and run* — next session you'll build from scratch.

By the end of this notebook you'll have used LangChain to:
1. Call a chat model and steer it with a system message
2. Turn prompts into reusable **templates**
3. Get **structured (JSON) output** you can trust
4. Parse model text into real Python objects
5. Compose everything into a **chain** — two different ways

**How to run:** click each cell and press **Shift+Enter**, top to bottom. That's it.

> Mental model for the whole bootcamp: **LangChain = the building blocks** (models, prompts, tools, parsers). **LangGraph = the graph that orchestrates them** — that's where we're headed.

## Step 0 — Setup (about 60 seconds)

Run the next two cells: install LangChain, then paste your OpenAI API key. Your key is entered with a hidden prompt and is **never saved** into the notebook.

In [1]:
# ⚙️ Run me first — installs LangChain (~30-60s)
!pip install -qU langchain langchain-openai langchain-core pydantic
print("✅ Installed. Run the next cell to add your API key.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 23.1 MB/s eta 0:00:00
✅ Installed. Run the next cell to add your API key.


In [2]:
import os
from getpass import getpass

# Paste your OpenAI API key when prompted (input stays hidden).
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("🔑 OPENAI_API_KEY: ")

# The one knob you'll change most often: which model powers everything below.
# Any current small chat model works — swap this name anytime.
MODEL = "gpt-4o-mini"
print(f"✅ Ready. Using model: {MODEL}")

🔑 OPENAI_API_KEY: ··········
✅ Ready. Using model: gpt-4o-mini


## 1 — Your first model call

LangChain gives you two interfaces to an LLM provider: the legacy **LLM** interface (plain text in, text out) and the **chat model** interface (messages with roles). Modern apps use **chat models**, so that's what we'll use everywhere.

`invoke()` sends input to the model and returns its prediction — here, an `AIMessage`.

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model = ChatOpenAI(model=MODEL)

response = model.invoke([HumanMessage("What is the capital of France?")])
print(type(response).__name__)   # AIMessage
print(response.content)          # The capital of France is Paris.

AIMessage
The capital of France is Paris.


### Roles: steering the model with a `SystemMessage`

Chat models split messages into roles:
- **System** — instructions the model should follow
- **Human (user)** — the user's query
- **AI (assistant)** — what the model generates

Watch how a system message changes the behaviour *without* changing the question.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage("You are a helpful assistant that answers every question with exactly three exclamation marks."),
    HumanMessage("What is the capital of France?"),
]

print(model.invoke(messages).content)   # e.g. Paris!!!

### One more knob: `temperature`

Temperature controls randomness. **Low (0.0)** = predictable and repeatable — good for structured or factual work. **High (~0.9)** = creative and varied. Run this a couple of times and watch the difference.

In [ ]:
creative = ChatOpenAI(model=MODEL, temperature=0.9)
precise  = ChatOpenAI(model=MODEL, temperature=0.0)

print("Creative:", creative.invoke("Describe the sky in five words.").content)
print("Precise: ", precise.invoke("Describe the sky in five words.").content)

## 2 — Prompts as reusable recipes

Hardcoding a prompt string works once. Real apps need prompts that change with the input. A **`ChatPromptTemplate`** is a recipe: it defines the structure and marks where dynamic values (in `{curly_braces}`) get filled in.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", 'Answer the question based only on the context below. '
               'If it cannot be answered from the context, say "I don\'t know".'),
    ("human", "Context: {context}"),
    ("human", "Question: {question}"),
])

# Filling the template gives you a concrete prompt (no model called yet):
prompt = template.invoke({
    "context": "LLMs are offered by OpenAI, Anthropic, and Cohere, "
               "plus open models you can run yourself.",
    "question": "Which providers offer LLMs?",
})

for m in prompt.messages:
    print(f"[{m.type}] {m.content}")

### Your first chain: `template | model`

The `|` (pipe) operator connects components — this is **LCEL**, the LangChain Expression Language. Read it left to right: *fill the template, then send it to the model.*

In [ ]:
chain = template | model

answer = chain.invoke({
    "context": "LLMs are offered by OpenAI, Anthropic, and Cohere, "
               "plus open models you can run yourself.",
    "question": "Which providers offer LLMs?",
})
print(answer.content)

## 3 — Structured output (JSON you can trust)

Plain text is fine for humans. But when the model's output feeds *other code*, you want a guaranteed shape. Define a schema with **Pydantic**, and `with_structured_output()` makes the model fill it — and validates the result before handing it back.

In [ ]:
from pydantic import BaseModel, Field

class AnswerWithJustification(BaseModel):
    """An answer to the user's question, with justification."""
    answer: str = Field(description="The answer to the user's question")
    justification: str = Field(description="Why the answer is correct")

structured_model = ChatOpenAI(model=MODEL, temperature=0).with_structured_output(AnswerWithJustification)

result = structured_model.invoke("What weighs more, a pound of bricks or a pound of feathers?")
print(type(result).__name__)     # AnswerWithJustification
print("answer:      ", result.answer)
print("justification:", result.justification)

## 4 — Output parsers

Sometimes you want a different machine-readable format — a list, CSV, XML. **Output parsers** turn the model's raw text into structured Python. They're most useful at the *end* of a chain.

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()
print(parser.invoke("apple, banana, cherry"))   # ['apple', 'banana', 'cherry']

# Put it at the end of a chain to get a real Python list back from the model:
list_prompt = ChatPromptTemplate.from_messages([
    ("human", "List 5 popular {topic}. Reply as a comma-separated list, nothing else."),
])
list_chain = list_prompt | model | parser
print(list_chain.invoke({"topic": "programming languages"}))

## 5 — One interface to rule them all: the Runnable

Notice every component so far — model, template, parser, chain — is used the *same way*. That's the **Runnable interface**, and it gives every block three verbs:

- **`invoke`** — one input → one output
- **`batch`** — many inputs → many outputs (efficiently)
- **`stream`** — one input → output as it's produced, token by token

In [ ]:
# invoke: single call
print("invoke →", model.invoke("Say hello in one word.").content)

# batch: many at once
print("batch  →", [r.content for r in model.batch(["Say hello", "Say goodbye"])])

# stream: token by token
print("stream →", end=" ")
for chunk in model.stream("Count from 1 to 5."):
    print(chunk.content, end="", flush=True)
print()

## 6 — Composing an app: imperative vs. declarative

Two ways to combine building blocks into an application.

**Imperative** — ordinary Python. Write a function, wrap it with `@chain`, and it gains the Runnable interface. Best when you have lots of custom logic.

In [ ]:
from langchain_core.runnables import chain as as_runnable

qa = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{question}"),
])

@as_runnable
def chatbot(values):
    prompt = qa.invoke(values)
    return model.invoke(prompt)

print(chatbot.invoke({"question": "Which providers offer LLMs?"}).content)

**Declarative** — LCEL with the `|` operator. Same result, less code, and you get streaming, batching, async, and tracing *for free*. Best for simply assembling existing components.

`StrOutputParser` at the end just pulls the plain string out of the `AIMessage`.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chatbot_lcel = qa | model | StrOutputParser()

print(chatbot_lcel.invoke({"question": "Which providers offer LLMs?"}))

## 🎉 You did it

In one notebook you went from a single model call to a composed LLM application. Every real LangChain app is a variation on this: **prompt → model → (optional) parser**, assembled as a chain.

### Recap
- **Chat models** take role-based messages; **system messages** steer behaviour.
- **Prompt templates** make prompts reusable with `{variables}`.
- **Structured output** (Pydantic) gives you validated JSON.
- **Output parsers** turn text into Python objects.
- Every block shares the **Runnable interface** — `invoke` / `batch` / `stream`.
- Compose **imperatively** (`@chain`) or **declaratively** (LCEL `|`).

### Your turn
Open the **practice assignment** and build a small chain of your own.

### Next session
We turn on **LangSmith** to *see* every step of these chains as a trace, then step up to **LangGraph** to orchestrate multi-step agents.

---
#### Optional — peek at a trace (next-session teaser)
Uncomment below, add a free [LangSmith](https://smith.langchain.com) key, then re-run any chain above and watch it appear step-by-step. Totally optional today.

In [ ]:
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = getpass("LANGSMITH_API_KEY: ")
# chatbot_lcel.invoke({"question": "Which providers offer LLMs?"})